### Main Exercise: Finding Similar Actors based on Genre

In [5]:
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
from sklearn.metrics import DistanceMetric

In [7]:
from pathlib import Path
possible_files = ["imdb_movies_2000to2022.prolific.json",]

file_path = None

for filename in possible_files:
    if Path(filename).exists():
        file_path = filename
        break

if file_path is None:
    raise FileNotFoundError(
        "IMDb JSON file not found. Upload it or place it in the same folder as the notebook."
    )

print("Using file:", file_path)

Using file: imdb_movies_2000to2022.prolific.json


In [8]:
# Outlier dictionary: actor ID
# Inner Dictionary: genre and number of appearences
actor_genre_counts = defaultdict(lambda: defaultdict(int))

# Maps actor ID to actor names
actor_id_to_name = {}

with open(file_path, "r", encoding="utf-8") as in_file:
    for line in in_file:
        line = line.strip()
        if not line:
            continue

        movie = json.loads(line)
        genres = movie.get("genres", [])
        actors = movie.get("actors", [])

        # Give every actor one appearence in each of thr movie's genres
        for actor_id, actor_name in actors:
            actor_id_to_name[actor_id] = actor_name
            for genre in genres:
                actor_genre_counts[actor_id][genre] += 1

In [9]:
actor_genre_df = pd.DataFrame.from_dict(
    actor_genre_counts, orient="index"
)

# Missing Genres appearences become zero
actor_genre_df = actor_genre_df.fillna(0).astype(int)

# Index Name
actor_genre_df.index.name = "actor_id"

print("Feature matrix shape:", actor_genre_df.shape)

actor_genre_df.head()

Feature matrix shape: (33609, 25)


,Comedy,Fantasy,Romance,Drama,Mystery,Thriller,Action,Biography,Crime,War,...,Horror,Documentary,Sport,News,Family,Music,,Western,Short,Reality-TV
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0000212,7,1,6,6,1,2,1,1,2,1,...,0,0,0,0,0,0,0,0,0,0
nm0413168,7,3,5,12,5,2,14,4,6,0,...,0,0,0,0,0,0,0,0,0,0
nm0000630,8,2,6,14,2,3,4,5,1,1,...,3,7,3,1,0,0,0,0,0,0
nm0005227,10,1,2,2,0,1,1,0,0,0,...,1,0,1,0,2,0,0,0,0,0
nm0864851,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [10]:
query_actor_id = "nm0000138"  # Leonardo DiCaprio

if query_actor_id not in actor_genre_df.index:
    raise ValueError(f"Actor ID {query_actor_id} not found in the dataset.")
print(f"Query Actor: {actor_id_to_name[query_actor_id]} (ID: {query_actor_id})")

# Double brackets keep the query as a 2D matrix
query_vector = actor_genre_df.loc[[query_actor_id]].to_numpy(dtype=float)

Query Actor: Leonardo DiCaprio (ID: nm0000138)


In [11]:
euclidean = DistanceMetric.get_metric("euclidean")
feature_matrix = actor_genre_df.to_numpy(dtype=float)

# Calculate distances from every actor to Leaonardo DiCaprio
distances = euclidean.pairwise(feature_matrix, query_vector).flatten()


In [14]:
results_df = pd.DataFrame({"actor_id": actor_genre_df.index, 
    "actor_name": [actor_id_to_name.get(actor_id, "Unknown") 
            for actor_id in actor_genre_df.index], "euclidean_distance": distances})

# Remove Lenardo DiCaprio himself because his distance is 0
results_df = results_df[results_df["actor_id"] != query_actor_id]

# Smaller distances mean more similar actors
top_10_similar_actors = results_df.sort_values("euclidean_distance").head(10).reset_index(drop=True)

print("Top 10 actors most similar to Leanardo DiCaprio:\n")
print(top_10_similar_actors.to_string(index=False))

Top 10 actors most similar to Leanardo DiCaprio:

 actor_id           actor_name  euclidean_distance
nm0000849        Javier Bardem            4.582576
nm0330687 Joseph Gordon-Levitt            5.099020
nm0266824       Dakota Fanning            5.196152
nm0578853       Ben Mendelsohn            5.196152
nm0000178           Diane Lane            5.196152
nm0001557      Viggo Mortensen            5.291503
nm0000368           Laura Dern            5.385165
nm0544718            Kate Mara            5.830952
nm0005476         Hilary Swank            5.830952
nm0000560           Nick Nolte            5.916080


### Additional Exercise: Finding Similar Actors Based On Co-stars

In [3]:
import numpy as np
import pandas as pd
import networkx as nx

from sklearn.metrics import DistanceMetric

In [33]:
# Keep a fixed actor order so the matrix rows and columns match
# Build co-star graph (nodes = actors, edges weighted by co-appearances)
g = nx.Graph()
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        movie = json.loads(line)
        actors_in_movie = movie.get("actors", [])
        actor_ids_in_movie = [aid for aid, _ in actors_in_movie]

        # ensure nodes exist (store name as attribute if available)
        for aid, name in actors_in_movie:
            if not g.has_node(aid):
                g.add_node(aid, name=name)

        # add / increment edge weights for every co-star pair
        for i in range(len(actor_ids_in_movie)):
            for j in range(i + 1, len(actor_ids_in_movie)):
                a = actor_ids_in_movie[i]
                b = actor_ids_in_movie[j]
                if g.has_edge(a, b):
                    g[a][b]["weight"] += 1
                else:
                    g.add_edge(a, b, weight=1)

# Keep a fixed actor order so the matrix rows and columns match
actor_ids = list(g.nodes())

# Create a weighted adjacency matrix
costar_matrix = nx.adjacency_matrix(g, nodelist=actor_ids, weight="weight")

# Convert the matrix into a Data Frame
actor_costar_df = pd.DataFrame(costar_matrix.toarray(), index=actor_ids, columns=actor_ids)
actor_costar_df.index.name = "actor_id"
print("Co-star Feature matrix shape:", actor_costar_df.shape)
actor_costar_df.head()

Co-star Feature matrix shape: (33609, 33609)


,nm0000212,nm0413168,nm0000630,nm0005227,nm0864851,nm0828288,nm0933983,nm0329491,nm0000417,nm0000603,...,nm3768164,nm6522322,nm9169920,nm1644256,nm10067359,nm9504284,nm10592896,nm7216750,nm0936300,nm10375007
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0000212,0,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0413168,1,0,2,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0000630,1,2,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0005227,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nm0864851,0,0,0,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [34]:
query_actor_id = "nm0424060"  # Scarlett Johansson

if query_actor_id not in actor_costar_df.index:
    raise ValueError(f"Actor ID {query_actor_id} not found in the dataset.")

query_actor_name = g.nodes[query_actor_id].get("name", "Unknown")
print(f"Query Actor: {query_actor_name} (ID: {query_actor_id})")

# Double brackets keep the query vector two-dimensional
query_vector = actor_costar_df.loc[[query_actor_id]].to_numpy(dtype=float)

Query Actor: Scarlett Johansson (ID: nm0424060)


In [35]:
# Calculating Euclidean distances
from sklearn.metrics import DistanceMetric
euclidean = DistanceMetric.get_metric("euclidean")
feature_matrix = actor_costar_df.to_numpy(dtype=float)

distances = euclidean.pairwise(feature_matrix, query_vector).flatten()
print("Number of distances calculated:", len(distances))

Number of distances calculated: 33609


In [36]:
# Creating the results of the DataFrame
results_df = pd.DataFrame({"actor_id": actor_costar_df.index,
    "actor_name": [g.nodes[actor_id].get("name", "Unknown")
            for actor_id in actor_costar_df.index], "euclidean_distance": distances})


In [37]:
# Finding the top 10
top_10_similar_actors = results_df.sort_values("euclidean_distance").head(10).reset_index(drop=True)
top_10_similar_actors

,actor_id,actor_name,euclidean_distance
0,nm0424060,Scarlett Johansson,0.000000
1,nm0812307,Peter Sohn,11.401754
2,nm10558382,Ricochet,11.489125
3,nm10309420,Dayna Hilton,11.489125
4,nm9125822,Max Ivutin,11.489125
5,nm0538683,Mako,11.489125
6,nm1812991,Tori Kelly,11.489125
7,nm6111880,Eva G. Cooper,11.575837
8,nm0606700,Kathryn Morris,11.575837
9,nm0661407,Judy Parfitt,11.575837
